# Public-data DINOv2 GON baseline

This notebook trains one explicit SSD or MSD configuration, selects the checkpoint using source validation only, then evaluates one unseen target. It is a public-data method reproduction, not an exact numerical reproduction of GONet.

Attach both Kaggle Datasets, select a GPU accelerator, and enable Internet for the first pretrained-weight download. The default run is a short GPU debug check.


In [ ]:
from pathlib import Path

# Edit this cell to choose the experiment and attached Kaggle input paths.
MODE = "SSD"
SOURCE_DOMAINS = ["HYRD"]
TARGET_DOMAIN = "REFUGE"
FAST_DEBUG = True
FULL_TRAIN = False
PRETRAINED = True
SEED = 42

INPUT_ROOT = Path("/kaggle/input")
CODE_INPUT = Path("/kaggle/input/glaucoma-dinov2-code")
DATA_INPUT = Path("/kaggle/input/glaucoma-public-data/data")
WORK_ROOT = Path("/kaggle/working/glaucoma_dinov2_workspace")

# These are updated automatically after the ZIPs are found/extracted.
PACKAGE_ROOT = WORK_ROOT
DATA_ROOT = DATA_INPUT
MANIFEST_PATH = CODE_INPUT / "manifests" / "all_public.csv"
CONFIG_PATH = CODE_INPUT / "configs" / "dinov2_baseline.yaml"
OUTPUT_DIR = WORK_ROOT / "outputs" / "dinov2_baseline"

if FULL_TRAIN:
    FAST_DEBUG = False
assert MODE in {"SSD", "MSD"}
if MODE == "SSD" and len(SOURCE_DOMAINS) != 1:
    raise ValueError("SSD requires exactly one source domain.")
if MODE == "MSD" and len(SOURCE_DOMAINS) < 2:
    raise ValueError("MSD requires at least two source domains.")
assert TARGET_DOMAIN not in SOURCE_DOMAINS
assert FAST_DEBUG ^ FULL_TRAIN, "Choose exactly one of FAST_DEBUG or FULL_TRAIN."


In [ ]:
import os, random, sys
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"device": str(device), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "seed": SEED})


In [ ]:
import subprocess, zipfile

WORK_ROOT.mkdir(parents=True, exist_ok=True)

def safe_extract(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Unsafe ZIP member path: {item.filename}")
        archive.extractall(destination)

code_zips = sorted(INPUT_ROOT.rglob("glaucoma_dinov2_code.zip")) if INPUT_ROOT.exists() else []
data_zips = sorted(INPUT_ROOT.rglob("glaucoma_public_data.zip")) if INPUT_ROOT.exists() else []
if code_zips:
    safe_extract(code_zips[0], WORK_ROOT)
if data_zips:
    safe_extract(data_zips[0], WORK_ROOT)

if (WORK_ROOT / "src" / "dinov2_engine.py").is_file():
    PACKAGE_ROOT = WORK_ROOT
elif (CODE_INPUT / "src" / "dinov2_engine.py").is_file():
    PACKAGE_ROOT = CODE_INPUT
else:
    engines = list(INPUT_ROOT.rglob("dinov2_engine.py")) if INPUT_ROOT.exists() else []
    if not engines:
        raise FileNotFoundError("Could not find the code ZIP or src/dinov2_engine.py in Kaggle inputs.")
    PACKAGE_ROOT = engines[0].parent.parent

if (WORK_ROOT / "data" / "HYDR").is_dir():
    DATA_ROOT = WORK_ROOT / "data"
elif DATA_INPUT.is_dir():
    DATA_ROOT = DATA_INPUT
else:
    hydr_labels = list(INPUT_ROOT.rglob("Labels.csv")) if INPUT_ROOT.exists() else []
    hydr_labels = [path for path in hydr_labels if path.parent.name == "HYDR"]
    if not hydr_labels:
        raise FileNotFoundError("Could not find the data ZIP or HYDR/Labels.csv in Kaggle inputs.")
    DATA_ROOT = hydr_labels[0].parent.parent

MANIFEST_PATH = PACKAGE_ROOT / "manifests" / "all_public.csv"
CONFIG_PATH = PACKAGE_ROOT / "configs" / "dinov2_baseline.yaml"
OUTPUT_DIR = WORK_ROOT / "outputs" / ("dinov2_" + MODE.lower() + "_" + TARGET_DOMAIN)
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(MANIFEST_PATH)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)

# Install only model/config/augmentation helpers; preserve Kaggle's CUDA torch build.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PACKAGE_ROOT / "requirements.txt")])
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
print({"package_root": str(PACKAGE_ROOT), "data_root": str(DATA_ROOT), "manifest": str(MANIFEST_PATH), "output_dir": str(OUTPUT_DIR), "code_zip_found": bool(code_zips), "data_zip_found": bool(data_zips)})


In [ ]:
import pandas as pd

manifest = pd.read_csv(MANIFEST_PATH, keep_default_na=False)
required = {"image_path", "domain", "label", "patient_id", "source_split"}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest missing columns: {sorted(missing)}")
manifest["label"] = pd.to_numeric(manifest["label"], errors="raise").astype(int)
assert set(manifest["label"].unique()).issubset({0, 1})
assert TARGET_DOMAIN in set(manifest["domain"])
assert set(SOURCE_DOMAINS).issubset(set(manifest["domain"]))
print(f"Rows: {len(manifest):,}; domains: {sorted(manifest.domain.unique().tolist())}")


In [ ]:
source_rows = manifest[manifest.domain.isin(SOURCE_DOMAINS)]
target_rows = manifest[manifest.domain == TARGET_DOMAIN]
if set(SOURCE_DOMAINS) & {TARGET_DOMAIN}:
    raise ValueError("Target domain must be absent from SOURCE_DOMAINS.")
source_summary = source_rows.groupby(["domain", "source_split", "label"]).size().unstack(fill_value=0)
display(source_summary.rename(columns={0: "GON-", 1: "GON+"}))
print(f"Unseen target rows available: {len(target_rows):,} (target labels withheld until final evaluation)")


In [ ]:
from dataset import ManifestDataset, build_manifest_loader, get_msd_transforms
from torch.utils.data import DataLoader

preview_rows = source_rows[source_rows.source_split == "train"].groupby("domain", group_keys=False).head(2).reset_index(drop=True)
preview_dataset = ManifestDataset(preview_rows, DATA_ROOT, transform=get_msd_transforms("val", 392))
preview_loader = DataLoader(preview_dataset, batch_size=min(8, len(preview_dataset)), shuffle=False, num_workers=0)


In [ ]:
import matplotlib.pyplot as plt

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
images, labels, paths = next(iter(preview_loader))
fig, axes = plt.subplots(1, len(paths), figsize=(4 * len(paths), 4))
axes = np.atleast_1d(axes)
for axis, tensor, label, path in zip(axes, images, labels, paths):
    row = manifest.loc[manifest.image_path == path].iloc[0]
    rgb = (tensor.permute(1, 2, 0).numpy() * std + mean).clip(0, 1)
    axis.imshow(rgb)
    axis.set_title(f"{row.domain} | {'GON+' if int(label) else 'GON-'}")
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
from models import DINO_MODEL, build_model

if device.type != "cuda":
    raise RuntimeError("Select a Kaggle GPU accelerator before running training; CPU fine-tuning is disabled in this notebook.")
model = build_model(
    model_name="gonet", pretrained=PRETRAINED, dropout_rate=0.0,
    device=str(device), image_size=392,
)
print("Pretrained backbone:", DINO_MODEL)


In [ ]:
total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print({"total_parameters": total_parameters, "trainable_parameters": trainable_parameters, "all_parameters_trainable": total_parameters == trainable_parameters})


In [ ]:
from types import SimpleNamespace

EPOCHS = 2 if FAST_DEBUG else 20
BATCH_SIZE = 4 if FAST_DEBUG else 16
NUM_WORKERS = 2
MAX_SAMPLES_PER_SPLIT = 8 if FAST_DEBUG else None
args = SimpleNamespace(
    config=str(CONFIG_PATH), manifest=str(MANIFEST_PATH), data_root=str(DATA_ROOT),
    output_dir=str(OUTPUT_DIR), epochs=EPOCHS, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, no_pretrained=not PRETRAINED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)
print({"mode": MODE, "sources": SOURCE_DOMAINS, "target": TARGET_DOMAIN, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "max_samples_per_split": MAX_SAMPLES_PER_SPLIT, "bootstrap": "1000 resamples of 95% target images"})


In [ ]:
from dinov2_engine import run_experiment

# The pretrained DINOv2 model built above is passed in, so its weights are not reloaded per epoch.
training_result = run_experiment(
    MODE.lower(), SOURCE_DOMAINS, TARGET_DOMAIN, args,
    model_instance=model, defer_target_evaluation=True,
)
print({key: value for key, value in training_result.items() if not key.startswith("_") and key != "history"})


In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
fig, axis = plt.subplots(figsize=(7, 4))
axis.plot(history.epoch, history.train_loss, marker="o", label="source train BCE")
axis.plot(history.epoch, history.val_loss, marker="o", label="source validation BCE")
axis.set(xlabel="Epoch", ylabel="BCE loss", title="Source-only training history")
axis.legend()
axis.grid(alpha=0.25)
plt.show()


In [ ]:
checkpoint_path = Path(training_result["checkpoint_path"])
try:
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
except TypeError:  # compatibility with older Kaggle torch versions
    checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.eval()
print({"loaded_checkpoint": str(checkpoint_path), "best_epoch": checkpoint["epoch"], "source_validation_loss": checkpoint["validation_loss"]})


In [ ]:
from dinov2_engine import evaluate_target_checkpoint

# This is the first cell that constructs a target loader or uses target labels.
metrics = evaluate_target_checkpoint(model, training_result["_evaluation_context"])
print({"target_domain": TARGET_DOMAIN, "n_target": metrics["n_target"], "roc_auc": metrics["roc_auc"], "brier_score": metrics["brier_score"]})


In [ ]:
from sklearn.metrics import RocCurveDisplay

predictions_path = OUTPUT_DIR / "predictions.csv"
predictions = pd.read_csv(predictions_path)
RocCurveDisplay.from_predictions(predictions.label, predictions.prob_gon)
plt.title(f"Unseen target ROC: {TARGET_DOMAIN}")
plt.grid(alpha=0.25)
plt.show()


In [ ]:
ci = metrics["roc_auc_bootstrap_95ci"]
print({
    "ROC-AUC": metrics["roc_auc"],
    "bootstrap_95_CI": [ci["low"], ci["high"]],
    "valid_bootstrap_samples": ci["valid_resamples"],
    "requested_bootstrap_samples": ci["requested_resamples"],
    "Brier": metrics["brier_score"],
    "target_n": metrics["n_target"],
})


In [ ]:
# Predictions are already written by the evaluation engine; show the required schema.
print("Predictions:", predictions_path)
display(predictions.head())


In [ ]:
metrics_path = OUTPUT_DIR / "metrics.json"
print("Metrics:", metrics_path)
print(metrics_path.read_text()[:4000])


## Optional: switch this same notebook to MSD

Change only the configuration cell at the top:

```python
MODE = "MSD"
SOURCE_DOMAINS = ["HYRD", "PAPILA", "DRISHTI_GS"]
TARGET_DOMAIN = "REFUGE"
FAST_DEBUG = True  # change to False and FULL_TRAIN=True for a full run
```

GAMMA is available as an explicit source if desired, but is excluded from the default list. Keep the target domain out of `SOURCE_DOMAINS`. Each run uses the same source-only selection and target evaluation path.
